# Pré-Processamento de Dados: Encoding, Divisão Temporal e Escalonamento

## 1. Contexto e Objetivos

Este notebook realiza o pipeline formal de pré-processamento para transformar a **Tabela Mestra Bruta** (`data/tabela_mestra_bruta.csv`) em matrizes numéricas prontas para treinamento e validação de modelos de Machine Learning (XGBoost, Random Forest, LightGBM, Regressão Logística).

### Etapas Executadas:
1. **Divisão Temporal Estrita (Train-Test Split Temporal):**
   * **Base de Treino:** Dados de 01/01/2023 a 31/12/2025 (3 anos de histórico operacional).
   * **Base de Teste (Validação Cega):** Dados de 01/01/2026 a 28/02/2026 (simulação de produção em tempo real).
2. **Engenharia de Encoding Categórico:**
   * **One-Hot Encoding (OHE):** Variável `estacao` (estações reais monitoradas).
   * **Binary Mapping:** Variável `plataforma` (`P1` = 0, `P2` = 1).
   * **Cyclical Encoding (Seno e Cosseno):** Variável `dia_da_semana` para preservar a continuidade circular do calendário.
3. **Normalização e Escalonamento Numérico:**
   * Aplicação do `RobustScaler` ajustado (*fit*) **estritamente na partição de Treino** para evitar *Data Leakage*.
4. **Exportação dos Datasets de Modelagem:**
   * `data/treino_pronto_modelagem.csv` e `data/teste_pronto_modelagem.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 120

print("✅ Bibliotecas de pré-processamento importadas com sucesso!")


## 2. Carregamento da Tabela Mestra Bruta

Carregamos a base consolidada gerada na etapa anterior e inspecionamos sua integridade e tipos de dados.


In [ ]:
path_mestra = "../../data/tabela_mestra_bruta.csv"
df = pd.read_csv(path_mestra)
df['data_ref'] = pd.to_datetime(df['data_ref'])

print(f"Dataset bruto carregado: {df.shape[0]:,} linhas e {df.shape[1]} colunas.")
print(f"Período temporal: de {df['data_ref'].min().strftime('%d/%m/%Y')} até {df['data_ref'].max().strftime('%d/%m/%Y')}.")
display(df.head())



## 3. Divisão Temporal Estrita (Train-Test Split Anti-Leakage)

Para séries temporais em manutenção preditiva, a divisão aleatória tradicional (*random shuffle*) é inadequada pois provocaria vazamento de informações futuras para o passado.

Aplicamos a divisão com ponto de corte no início de 2026:
* **Treino:** 01/01/2023 a 31/12/2025
* **Teste:** 01/01/2026 a 28/02/2026


In [ ]:
data_corte = pd.to_datetime('2026-01-01')

df_treino = df[df['data_ref'] < data_corte].copy()
df_teste = df[df['data_ref'] >= data_corte].copy()

print(f"Partição de TREINO (2023-2025): {df_treino.shape[0]:,} linhas ({(len(df_treino)/len(df))*100:.1f}%)")
print(f"  - Target=1 no Treino: {df_treino['target_falha_7d'].sum():,} ({df_treino['target_falha_7d'].mean()*100:.2f}%)")

print(f"\nPartição de TESTE (2026): {df_teste.shape[0]:,} linhas ({(len(df_teste)/len(df))*100:.1f}%)")
print(f"  - Target=1 no Teste: {df_teste['target_falha_7d'].sum():,} ({df_teste['target_falha_7d'].mean()*100:.2f}%)")



## 4. Aplicação das Técnicas de Encoding

### 4.1 One-Hot Encoding em Estações
Como as estações são nominais e possuem volumes de falhas variados, geramos colunas binárias independentes:

### 4.2 Binary Mapping em Plataforma
`P1` -> 0, `P2` -> 1.

### 4.3 Cyclical Encoding em Dia da Semana
* `dia_semana_sin = sin(2 * pi * dia / 7)`
* `dia_semana_cos = cos(2 * pi * dia / 7)`


In [ ]:
def aplicar_encodings(df_in):
    df_out = df_in.copy()
    
    # 1. Cyclical Encoding para Dia da Semana
    df_out['dia_semana_sin'] = np.sin(2 * np.pi * df_out['dia_semana_num'] / 7.0)
    df_out['dia_semana_cos'] = np.cos(2 * np.pi * df_out['dia_semana_num'] / 7.0)
    
    # 2. Binary Mapping para Plataforma
    df_out['plataforma_bin'] = df_out['plataforma'].map({'P1': 0, 'P2': 1}).fillna(0).astype(int)
    
    # 3. One-Hot Encoding para Estações
    df_out = pd.get_dummies(df_out, columns=['estacao'], prefix='estacao', drop_first=False)
    
    return df_out

print("Aplicando encodings nas partições de treino e teste...")
df_treino_enc = aplicar_encodings(df_treino)
df_teste_enc = aplicar_encodings(df_teste)

# Alinhamento das colunas (garante as mesmas colunas exatas nas duas partições)
df_treino_enc, df_teste_enc = df_treino_enc.align(df_teste_enc, join='left', axis=1, fill_value=0)

print(f"Novas dimensões pós-encoding:")
print(f"  Treino: {df_treino_enc.shape}")
print(f"  Teste:  {df_teste_enc.shape}")
display(df_treino_enc[['id_ativo', 'data_ref', 'plataforma_bin', 'dia_semana_sin', 'dia_semana_cos', 'estacao_REP', 'estacao_BUT']].head())



## 5. Escalonamento e Normalização das Features Numéricas

As variáveis de contagem de alarmes, tempos de normalização e dias de manutenção possuem escalas muito distintas (de 0 a dezenas de milhares de segundos).

Utilizamos o **`RobustScaler`** (que utiliza a mediana e o intervalo interquartil IQR), sendo mais robusto a picos esporádicos de paradas do que o StandardScaler tradicional.

> [!IMPORTANT]
> O escalonador é ajustado (`fit`) **estritamente na partição de Treino** e apenas aplicado (`transform`) na partição de Teste.


In [ ]:
cols_para_escalonar = [
    'idade_ativo_anos', 'porta',
    'qtd_alarmes_1d', 'qtd_alarmes_7d', 'qtd_alarmes_14d',
    'qtd_anomalias_30s_1d', 'qtd_anomalias_30s_7d', 'qtd_anomalias_30s_14d',
    'tempo_parada_acumulado_7d', 'max_tempo_normalizacao_7d',
    'qtd_alarmes_rce_7d', 'dias_desde_ultima_zm01', 'dias_desde_ultima_zm02', 'dias_desde_ultima_zm03'
]

print("Ajustando RobustScaler na partição de Treino...")
scaler = RobustScaler()
scaler.fit(df_treino_enc[cols_para_escalonar])

cols_escalonadas = [f"{c}_scaled" for c in cols_para_escalonar]

df_treino_enc[cols_escalonadas] = scaler.transform(df_treino_enc[cols_para_escalonar])
df_teste_enc[cols_escalonadas] = scaler.transform(df_teste_enc[cols_para_escalonar])

# Salvar o objeto scaler para reutilização em inferência
os.makedirs("../../data/modelos_artefatos", exist_ok=True)
joblib.dump(scaler, "../../data/modelos_artefatos/robust_scaler_features.joblib")
print("✅ Scaler ajustado e salvo em data/modelos_artefatos/robust_scaler_features.joblib!")

display(df_treino_enc[['id_ativo', 'data_ref', 'qtd_alarmes_7d_scaled', 'dias_desde_ultima_zm01_scaled']].head())



## 6. Organização das Matrizes de Features ($X$) e Target ($y$)

Identificamos todas as colunas de estações One-Hot geradas e organizamos as features prontas para consumo por modelos de Machine Learning.


In [ ]:
cols_estacoes = [c for c in df_treino_enc.columns if c.startswith('estacao_')]

features_modelagem = [
    # Categóricas Codificadas
    'plataforma_bin', 'flag_fim_de_semana', 'dia_semana_sin', 'dia_semana_cos'
] + cols_estacoes + [
    # Numéricas Escalonadas
    'idade_ativo_anos_scaled', 'porta_scaled',
    'qtd_alarmes_1d_scaled', 'qtd_alarmes_7d_scaled', 'qtd_alarmes_14d_scaled',
    'qtd_anomalias_30s_1d_scaled', 'qtd_anomalias_30s_7d_scaled', 'qtd_anomalias_30s_14d_scaled',
    'tempo_parada_acumulado_7d_scaled', 'max_tempo_normalizacao_7d_scaled',
    'qtd_alarmes_rce_7d_scaled',
    'dias_desde_ultima_zm01_scaled', 'dias_desde_ultima_zm02_scaled', 'dias_desde_ultima_zm03_scaled'
]

target_coluna = 'target_falha_7d'

print(f"Total de Features Preditivas Selecionadas: {len(features_modelagem)}")
print("Lista de Features:")
for i, f in enumerate(features_modelagem, 1):
    print(f"  {i:02d}. {f}")



## 7. Exportação dos Datasets Finais de Treino e Teste

Salvamos as bases de Treino e Teste pré-processadas em formato CSV.


In [ ]:
path_treino_out = "../../data/treino_pronto_modelagem.csv"
path_teste_out = "../../data/teste_pronto_modelagem.csv"
path_total_out = "../../data/tabela_mestra_encoded.csv"

# Exportação mantendo os metadados de identificação + features + target
cols_exportacao = ['id_ativo', 'data_ref'] + features_modelagem + [target_coluna]

df_treino_enc[cols_exportacao].to_csv(path_treino_out, index=False, encoding='utf-8')
df_teste_enc[cols_exportacao].to_csv(path_teste_out, index=False, encoding='utf-8')

# Dataset total combinado encodado
df_total_enc = pd.concat([df_treino_enc[cols_exportacao], df_teste_enc[cols_exportacao]], ignore_index=True)
df_total_enc.to_csv(path_total_out, index=False, encoding='utf-8')

print("✅ Datasets de modelagem exportados com sucesso!")
print(f"Treino salvo em: {path_treino_out} ({df_treino_enc.shape[0]:,} linhas, {len(cols_exportacao)} colunas)")
print(f"Teste salvo em:  {path_teste_out} ({df_teste_enc.shape[0]:,} linhas, {len(cols_exportacao)} colunas)")
print(f"Total salvo em:  {path_total_out} ({df_total_enc.shape[0]:,} linhas)")

